In [46]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [47]:
data = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train Shape:", data.shape)
print("Test Shape:", test.shape)

Train Shape: (1460, 81)
Test Shape: (1459, 80)


In [48]:
num_cols = data.select_dtypes(include=np.number).columns
cat_cols = data.select_dtypes(include="object").columns

for col in num_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in cat_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

In [49]:
num_cols_test = test.select_dtypes(include=np.number).columns
cat_cols_test = test.select_dtypes(include="object").columns

for col in num_cols_test:
    test[col] = test[col].fillna(test[col].mean())

for col in cat_cols_test:
    test[col] = test[col].fillna(test[col].mode()[0])

In [50]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]

# Keep ID for submission later
test_ids = test["Id"]

In [51]:
X = pd.get_dummies(X)
test = pd.get_dummies(test)

# Align test columns with train columns
test = test.reindex(columns=X.columns, fill_value=0)

In [52]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
test_processed = scaler.transform(test)

In [53]:
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

val_predictions = model.predict(X_val)

In [54]:
test_predictions = model.predict(test_processed)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_predictions
})

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")

Submission file created successfully!
